# TrafficFlowBench 2026 — 完整提交流水线

2026 IEEE Big Data Cup · Kaggle `2026-ieee-big-data-traffic-flow-bench`

这个 notebook 从比赛数据直接跑出 `submission.csv`（6 列的上传格式）。

**设计原则：不硬编码列名。** 比赛数据的实际列清单与本仓库文档可能有出入，
所以这里用 `pyarrow` 先读 parquet 的 schema，再按「我想要 ∩ 实际有」取列，
缺列时自动降级。这样即使某个文件少几列，流程也不会崩。

**三个任务的思路**

| 任务 | 方法 |
|---|---|
| Task 1 状态重建 | 时序插值 + 空间插值 + 历史均值，三路按可用性加权融合 |
| Task 2 排队预测 | 已在排队→持续性；即将排队→速度接近阈值的路段判为会排队 |
| Task 4 需求反推 | 带先验正则的非负最小二乘（NNLS） |

Task 3 没有单独提交，它直接读 Task 1 的答案来评分。

> 时长：全量 10 条走廊、train 全量档案，Kaggle CPU 上大约 40–90 分钟。
> 想先快速验证流程，把 `CFG["panels"]` 设成 `["D12_I5_N"]`。


In [ ]:
# ============================================================
# 0. 配置与工具
# ============================================================
import os
import gc
import glob
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)

CFG = {
    # None = 全部 10 条走廊；调试时设 ["D12_I5_N"]
    "panels": None,

    # 要产出哪些 split 的预测（submission_key.csv 里出现哪个就交哪个）
    "splits": ["validation", "private"],

    # 用多少个 train 日文件建历史档案。0 = 全用（273 天）
    # 120 天已足够稳定，且能省掉一半以上的读取时间
    "profile_max_files": 120,

    # 三路融合权重（会按实际可用性重新归一化）
    "w_temporal": 0.55,
    "w_spatial": 0.20,
    "w_historical": 0.25,

    "temporal_limit": 12,   # 时序方向最多连续插值 12 步（=60 分钟）
    "spatial_limit": 4,     # 空间方向最多连续插值 4 个路段

    # Task 2：onset 窗口里，速度低于 v_cut 的多少倍就判为「即将排队」
    "onset_factor": 1.15,

    # Task 4：先验正则系数，官方基线用 0.05
    "odme_lambda": 0.05,

    "out_path": "/kaggle/working/submission.csv",
    "work_dir": "/kaggle/working",
}

T0 = time.time()


def log(*args):
    print(f"[{time.time() - T0:7.1f}s]", *args, flush=True)


In [ ]:
# ============================================================
# 1. 定位数据根目录
# ============================================================
# Kaggle 上比赛数据挂在 /kaggle/input/<比赛slug>/ 下。
# 解压后应得到一个 kaggle_public/ 目录，里面才有 corridors/ 和 config/。
# 这里不写死路径，而是向下搜几层，找到含 corridors/ 的目录。

def find_release_root(bases=None, max_depth=3):
    """返回数据根目录，**类型是 pathlib.Path**。

    必须返回 Path 而不是 str：后面全部代码都用 `ROOT / "config" / ...` 这种写法，
    str 参与 `/` 运算会直接 TypeError。
    """
    if bases is None:
        bases = ("/kaggle/input", "/kaggle/working", os.getcwd())
    roots = []
    for base in bases:
        if not os.path.isdir(base):
            continue
        level = [base]
        for _ in range(max_depth):
            nxt = []
            for b in level:
                for d in glob.glob(os.path.join(b, "*")):
                    if os.path.isdir(d):
                        if os.path.isdir(os.path.join(d, "corridors")):
                            roots.append(d)
                        nxt.append(d)
            level = nxt
        if roots:
            break
    if not roots:
        raise FileNotFoundError(
            "找不到数据根目录（含 corridors/ 的目录）。\n"
            "请确认已把比赛数据添加到 notebook 的 Input 里。\n"
            f"已检查：{bases} 下前 {max_depth} 层"
        )
    # 多个候选时，优先名字里带 kaggle_public 的
    roots.sort(key=lambda p: (0 if "kaggle_public" in p else 1, len(p)))
    return Path(roots[0])


ROOT = find_release_root()
log("数据根目录:", ROOT)
log("顶层内容:", sorted(os.listdir(ROOT))[:12])


In [ ]:
# ============================================================
# 2. 通用工具：按实际 schema 取列、判定合格格
# ============================================================

def parquet_columns(path):
    """只读 parquet 的 footer 拿列名，不加载数据。"""
    import pyarrow.parquet as pq
    return list(pq.ParquetFile(path).schema.names)


def read_parquet_smart(path, wanted):
    """读 parquet，只取「想要 ∩ 实际有」的列。"""
    have = set(parquet_columns(path))
    cols = [c for c in wanted if c in have]
    if not cols:
        return pd.DataFrame()
    return pd.read_parquet(path, columns=cols)


def eligible_filter(df):
    """合格格过滤：优先用 is_score_eligible，退而用 pct_observed>=75，都没有就全算合格。"""
    if "is_score_eligible" in df.columns:
        return df[df["is_score_eligible"].astype(bool)]
    if "pct_observed" in df.columns:
        ok = pd.to_numeric(df["pct_observed"], errors="coerce") >= 75
        return df[ok.fillna(False)]
    return df


def as_link_str(df):
    if "link_id" in df.columns:
        df = df.copy()
        df["link_id"] = df["link_id"].astype(str)
    return df


def as_ts(df, col="timestamp"):
    if col in df.columns:
        df = df.copy()
        df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
    return df


In [ ]:
# ============================================================
# 3. 路网参数：自由流速度 / 通行能力 / 车道数 / 排队阈值
# ============================================================

def load_csv_safe(path, **kw):
    return pd.read_csv(path, **kw) if os.path.exists(path) else pd.DataFrame()


def network_params(panel_dir):
    """合并 links.csv 与 fd_parameters.csv，得到每个路段的物理参数。

    fd_parameters.csv 里的值是实测的（速度 99 分位、流量 99.9 分位），
    links.csv 的自由流速度是 OSM 的默认值 105，所以 fd_parameters 优先。
    """
    net = panel_dir / "network"
    links = as_link_str(load_csv_safe(net / "links.csv"))
    fd = as_link_str(load_csv_safe(net / "fd_parameters.csv"))

    out = pd.DataFrame()
    if len(links):
        keep = [c for c in ("link_id", "length_km", "free_speed_kmh", "capacity_vph", "lanes")
                if c in links.columns]
        out = links[keep].drop_duplicates("link_id").copy()
    if not len(out) and len(fd):
        out = fd[["link_id"]].drop_duplicates().copy()
    if not len(out):
        return pd.DataFrame(columns=["link_id", "free_speed_kmh", "capacity_vph", "lanes", "v_cut"])

    # fd_parameters 覆盖
    if len(fd):
        fcols = [c for c in ("link_id", "free_speed_kmh", "capacity_vph", "lanes", "k_jam", "v_cut")
                 if c in fd.columns]
        if len(fcols) > 1:
            agg = fd[fcols].groupby("link_id", as_index=False).mean(numeric_only=True)
            out = out.merge(agg, on="link_id", how="left", suffixes=("", "_fd"))
            for c in ("free_speed_kmh", "capacity_vph", "lanes"):
                if f"{c}_fd" in out.columns:
                    out[c] = out[f"{c}_fd"].fillna(out.get(c))
                    out = out.drop(columns=[f"{c}_fd"])
            out = out.drop(columns=[c for c in ("k_jam_merge",) if c in out.columns])

    # 兜底
    if "lanes" not in out.columns:
        out["lanes"] = 4.0
    out["lanes"] = pd.to_numeric(out["lanes"], errors="coerce").fillna(4.0).clip(lower=1.0)
    if "free_speed_kmh" not in out.columns:
        out["free_speed_kmh"] = 105.0
    out["free_speed_kmh"] = pd.to_numeric(out["free_speed_kmh"], errors="coerce").fillna(105.0)
    if "capacity_vph" not in out.columns:
        out["capacity_vph"] = np.nan
    out["capacity_vph"] = pd.to_numeric(out["capacity_vph"], errors="coerce")
    out["capacity_vph"] = out["capacity_vph"].fillna(1800.0 * out["lanes"])

    # 排队阈值：优先用文件里的 v_cut，否则 0.6 × 自由流速度
    if "v_cut" in out.columns:
        vc = pd.to_numeric(out["v_cut"], errors="coerce")
        out["v_cut"] = vc.fillna(0.6 * out["free_speed_kmh"])
    else:
        out["v_cut"] = 0.6 * out["free_speed_kmh"]
    return out


def link_mileposts(panel_dir, split="validation"):
    """尽力拿到 link_id -> milepost。没有就返回 None，后续按字典序排。"""
    for split_dir in ("train", "validation", "private"):
        files = sorted((panel_dir / split_dir / "mainline_states_masked").glob("**/*.parquet"))
        for p in files[:2]:
            d = read_parquet_smart(p, ["link_id", "milepost"])
            if len(d) and {"link_id", "milepost"}.issubset(d.columns):
                d = d.copy()
                d["link_id"] = d.link_id.astype(str)
                d["milepost"] = pd.to_numeric(d.milepost, errors="coerce")
                d = d.dropna().drop_duplicates("link_id")
                if len(d):
                    return d
    return None


In [ ]:
# ============================================================
# 4. Task 1 第一步：用 train 建历史档案
# ============================================================
# 每个路段 × 星期几 × 一天 288 个时间片 → 速度/流量的历史均值。
# 这就是官方「历史均值基线」的做法，也是融合里的兜底项。

N_SLOTS = 7 * 288


def build_profiles(panel_dir, cfg):
    train_dir = panel_dir / "train" / "mainline_states"
    files = sorted(train_dir.glob("**/*.parquet"))
    if not files:
        # 有些发布把未掩码层放在 mainline_states_masked 上层，做个兜底
        files = sorted((panel_dir / "train").glob("**/mainline_states/**/*.parquet"))
    if not files:
        log(f"    [警告] {panel_dir.name}: train 里没有未掩码观测层，历史档案将为空")
        return None, None, []

    if cfg["profile_max_files"] and len(files) > cfg["profile_max_files"]:
        # 均匀抽样，避免只取到某几个月
        idx = np.linspace(0, len(files) - 1, cfg["profile_max_files"]).astype(int)
        files = [files[i] for i in sorted(set(idx))]
    log(f"    读 {len(files)} 个 train 文件建档案")

    speed_sum, flow_sum, cnt = {}, {}, {}
    link_ids = None
    for i, p in enumerate(files, 1):
        d = read_parquet_smart(p, ["timestamp", "link_id", "speed_kmh", "flow_vph",
                                   "is_score_eligible", "pct_observed"])
        if not len(d) or "link_id" not in d.columns or "timestamp" not in d.columns:
            continue
        d = as_link_str(as_ts(d))
        d = eligible_filter(d)
        if not len(d):
            continue
        ts = d["timestamp"]
        d = d.assign(slot=(ts.dt.weekday * 288 + ts.dt.hour * 12 + ts.dt.minute // 5).to_numpy())
        g = d.groupby(["link_id", "slot"], as_index=False).agg(
            s=("speed_kmh", "mean"), f=("flow_vph", "mean"), n=("link_id", "size"))
        # name=None 取纯元组，避免 namedtuple 对下划线列名的重命名坑
        for lk, sl, sv, fv, nn in g[["link_id", "slot", "s", "f", "n"]].itertuples(index=False, name=None):
            speed_sum.setdefault(lk, np.zeros(N_SLOTS))[sl] += (0.0 if pd.isna(sv) else sv)
            flow_sum.setdefault(lk, np.zeros(N_SLOTS))[sl] += (0.0 if pd.isna(fv) else fv)
            cnt.setdefault(lk, np.zeros(N_SLOTS))[sl] += nn
        if i % 40 == 0:
            log(f"      档案进度 {i}/{len(files)}")
        del d

    if not cnt:
        return None, None, []

    link_ids = sorted(cnt.keys())
    prof_s = np.full((len(link_ids), N_SLOTS), np.nan)
    prof_f = np.full((len(link_ids), N_SLOTS), np.nan)
    for j, lk in enumerate(link_ids):
        c = cnt[lk]
        m = c > 0
        prof_s[j, m] = speed_sum[lk][m] / c[m]
        prof_f[j, m] = flow_sum[lk][m] / c[m]
    log(f"    档案完成：{len(link_ids)} 个路段")
    return prof_s, prof_f, link_ids


In [ ]:
# ============================================================
# 5. Task 1 第二步：三路融合预测
# ============================================================

def _blend(mats, weights):
    """按可用性加权平均。mats 是若干同形状数组，NaN 视为不可用。"""
    stack = np.stack(mats, axis=0).astype(np.float64)
    w = np.asarray(weights, dtype=np.float64).reshape(-1, 1, 1)
    ok = np.isfinite(stack)
    num = np.nansum(np.where(ok, stack, 0.0) * w, axis=0)
    den = (ok * w).sum(axis=0)
    out = np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan)
    return out


def _interp(df, axis, limit):
    """沿指定轴线性插值，并对 limit 做自适应限幅。

    pandas 在 limit 大于等于该轴长度时会抛
    "window shape cannot be larger than input array shape"，
    所以先把 limit 压到 n-1；万一仍然失败（不同 pandas 版本行为不一致），
    就退化成不带 limit 的插值，再不行原样返回，绝不让整个 Task 1 挂掉。
    """
    n = df.shape[axis]
    if n < 2:
        return df
    lim = None if not limit else max(1, min(int(limit), n - 1))
    try:
        return df.interpolate(method="linear", axis=axis, limit=lim, limit_direction="both")
    except Exception:
        try:
            return df.interpolate(method="linear", axis=axis, limit_direction="both")
        except Exception:
            return df


def predict_split(panel_dir, split, net, prof_s, prof_f, prof_links, milepost, cfg):
    """为某个 split 的每个目标格生成速度和流量预测。"""
    mdir = panel_dir / split / "mainline_states_masked"
    files = sorted(mdir.glob("**/*.parquet"))
    if not files:
        return None
    log(f"    {split}: 读 {len(files)} 个掩码文件")

    frames = []
    for p in files:
        d = read_parquet_smart(p, ["timestamp", "link_id", "speed_kmh", "flow_vph",
                                   "is_score_eligible", "pct_observed"])
        if not len(d) or "link_id" not in d.columns or "timestamp" not in d.columns:
            continue
        d = as_link_str(as_ts(d))
        d = eligible_filter(d)
        if len(d):
            frames.append(d[["timestamp", "link_id", "speed_kmh", "flow_vph"]])
    if not frames:
        return None

    obs = pd.concat(frames, ignore_index=True)
    del frames
    # 掩码是在「路段」层级做的（同一路段上多个检测器站一起被挖空），
    # 所以先把同一时刻同一路段的观测取平均，作为该路段的锚点。
    obs = obs.groupby(["timestamp", "link_id"], as_index=False)[["speed_kmh", "flow_vph"]].mean()

    S = obs.pivot(index="timestamp", columns="link_id", values="speed_kmh").sort_index()
    F = obs.pivot(index="timestamp", columns="link_id", values="flow_vph").sort_index()
    cols = sorted(set(S.columns) | set(F.columns))
    S = S.reindex(columns=cols)
    F = F.reindex(columns=cols)
    log(f"    {split}: 观测矩阵 {S.shape[0]} 时刻 × {S.shape[1]} 路段")

    # 按里程排序，让「空间插值」沿真实上下游方向进行
    if milepost is not None and len(milepost):
        order = [l for l in milepost.sort_values("milepost")["link_id"].tolist() if l in set(cols)]
        order += [c for c in cols if c not in set(order)]
        S = S[order]
        F = F[order]
        cols = order

    # --- 第 1 路：时序插值（同一路段沿时间） ---
    S_t = _interp(S, 0, cfg["temporal_limit"])
    F_t = _interp(F, 0, cfg["temporal_limit"])

    # --- 第 2 路：空间插值（同一时刻沿路段） ---
    S_s = _interp(S, 1, cfg["spatial_limit"])
    F_s = _interp(F, 1, cfg["spatial_limit"])

    # --- 第 3 路：历史档案 ---
    idx = S.index
    slot = (idx.weekday * 288 + idx.hour * 12 + idx.minute // 5).to_numpy()
    S_h = F_h = None
    if prof_s is not None and len(prof_links):
        pos = {l: i for i, l in enumerate(prof_links)}
        pi = np.array([pos.get(c, -1) for c in cols])
        good = pi >= 0
        S_h = np.full((len(idx), len(cols)), np.nan)
        F_h = np.full((len(idx), len(cols)), np.nan)
        if good.any():
            S_h[:, good] = prof_s[np.ix_(pi[good], slot)].T
            F_h[:, good] = prof_f[np.ix_(pi[good], slot)].T

    # --- 融合 ---
    w = [cfg["w_temporal"], cfg["w_spatial"]]
    mats_s = [S_t.to_numpy(dtype=float), S_s.to_numpy(dtype=float)]
    mats_f = [F_t.to_numpy(dtype=float), F_s.to_numpy(dtype=float)]
    if S_h is not None:
        w.append(cfg["w_historical"])
        mats_s.append(S_h)
        mats_f.append(F_h)
    pred_s = _blend(mats_s, w)
    pred_f = _blend(mats_f, w)

    # 完全无锚点的格子：退到路段历史均值，再退到全局中位数
    if S_h is not None:
        fall_s = np.nanmean(S_h, axis=0)
        fall_f = np.nanmean(F_h, axis=0)
    else:
        fall_s = np.nanmean(pred_s, axis=0)
        fall_f = np.nanmean(pred_f, axis=0)
    gs = np.nanmedian(fall_s) if np.isfinite(fall_s).any() else 90.0
    gf = np.nanmedian(fall_f) if np.isfinite(fall_f).any() else 1500.0
    fall_s = np.where(np.isfinite(fall_s), fall_s, gs)
    fall_f = np.where(np.isfinite(fall_f), fall_f, gf)
    bad_s = ~np.isfinite(pred_s)
    bad_f = ~np.isfinite(pred_f)
    pred_s[bad_s] = np.take(fall_s, np.where(bad_s)[1])
    pred_f[bad_f] = np.take(fall_f, np.where(bad_f)[1])

    # --- 物理裁剪 ---
    neti = net.set_index("link_id").reindex(cols)
    vf = pd.to_numeric(neti["free_speed_kmh"], errors="coerce").fillna(105.0).to_numpy()
    cap = pd.to_numeric(neti["capacity_vph"], errors="coerce").fillna(7200.0).to_numpy()
    pred_s = np.clip(pred_s, 5.0, np.maximum(1.05 * vf, 10.0))
    pred_f = np.clip(pred_f, 0.0, np.maximum(1.15 * cap, 100.0))

    return {"index": idx, "cols": cols, "speed": pred_s, "flow": pred_f}


def apply_state_predictions(root, panel, split, pred, cfg):
    """把矩阵预测映射回 sample_submission_state.csv 的每一行。"""
    tmpl_path = root / "task1" / panel / split / "sample_submission_state.csv"
    if not pred or not tmpl_path.exists():
        return None
    tmpl = pd.read_csv(tmpl_path, dtype=str)
    need = {"panel", "timestamp", "station_id", "link_id", "mask_regime"}
    if not need.issubset(tmpl.columns):
        log(f"    [警告] {panel}/{split} 模板缺列: {sorted(need - set(tmpl.columns))}")
        return None
    tmpl = as_link_str(as_ts(tmpl))

    tpos = pd.Series(np.arange(len(pred["index"])), index=pred["index"])
    lpos = pd.Series(np.arange(len(pred["cols"])), index=pred["cols"])
    ti = tmpl["timestamp"].map(tpos)
    li = tmpl["link_id"].map(lpos)
    ok = ti.notna() & li.notna()

    out = tmpl.loc[ok, ["panel", "timestamp", "station_id", "link_id", "mask_regime"]].copy()
    ti = ti[ok].astype(int).to_numpy()
    li = li[ok].astype(int).to_numpy()
    out["speed_kmh"] = pred["speed"][ti, li]
    out["flow_vph"] = pred["flow"][ti, li]
    # 时间戳写回原始文本形式（掩码是对文本做哈希的，格式必须一致）
    out["timestamp"] = out["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    miss = int((~ok).sum())
    if miss:
        log(f"    [提示] {panel}/{split}: {miss:,} 行模板在观测里没有对应时刻/路段，已跳过")
    return out


In [ ]:
# ============================================================
# 6. Task 2：排队预测
# ============================================================

def build_queue_predictions(root, panel, split, net, cfg):
    tdir = root / "task2" / panel / split
    tmpl_path = tdir / "sample_submission_queue.csv"
    hist_path = tdir / "window_history.parquet"
    widx_path = tdir / "window_index.csv"
    if not (tmpl_path.exists() and hist_path.exists()):
        return None

    tmpl = pd.read_csv(tmpl_path, dtype=str)
    if not {"window_id", "timestamp", "link_id"}.issubset(tmpl.columns):
        return None
    tmpl = as_link_str(tmpl)
    tmpl["window_id"] = tmpl.window_id.astype(str)

    hist = read_parquet_smart(hist_path, ["window_id", "timestamp", "link_id",
                                          "speed_kmh", "is_score_eligible"])
    if not len(hist):
        return None
    hist = as_link_str(as_ts(hist))
    hist["window_id"] = hist.window_id.astype(str)
    hist["speed_kmh"] = pd.to_numeric(hist["speed_kmh"], errors="coerce")

    cond_by_window = {}
    if widx_path.exists():
        w = pd.read_csv(widx_path, dtype=str)
        if "window_id" in w.columns and "condition" in w.columns:
            cond_by_window = dict(zip(w.window_id.astype(str), w.condition.astype(str)))

    vcut = dict(zip(net["link_id"].astype(str), net["v_cut"].astype(float)))

    rows = []
    wanted_windows = set(tmpl.window_id.astype(str))
    for wid, h in hist.groupby("window_id", sort=False):
        if wid not in wanted_windows:
            continue
        last = h["timestamp"].max()
        at = h[h["timestamp"] == last]
        if "is_score_eligible" in at.columns:
            at = at[at["is_score_eligible"].astype(bool)]
        cut = at["link_id"].map(vcut).fillna(63.0).astype(float)
        queued_now = dict(zip(at["link_id"], (at["speed_kmh"] <= cut).fillna(False)))
        cond = cond_by_window.get(wid, "queue_ongoing")

        t = tmpl[tmpl.window_id == wid][["window_id", "timestamp", "link_id"]].copy()
        if cond == "queue_onset":
            # 「即将排队」：现在还没排，但速度已经贴近阈值 → 判为接下来会排队
            near = at.assign(_cut=cut)
            near = near[near["speed_kmh"] <= near["_cut"] * cfg["onset_factor"]]
            flag = dict(zip(near["link_id"], [1] * len(near)))
        else:
            # 「已在排队」：持续性假设
            flag = {k: int(bool(v)) for k, v in queued_now.items()}
        t["queue_pred"] = t["link_id"].map(flag).fillna(0).astype(int)
        rows.append(t)

    if not rows:
        return None
    out = pd.concat(rows, ignore_index=True).drop_duplicates(["window_id", "timestamp", "link_id"])
    out["timestamp"] = out["timestamp"].astype(str)
    return out


In [ ]:
# ============================================================
# 7. Task 4：带先验正则的非负最小二乘（ODME）
# ============================================================

def _nnls(A, b):
    """非负最小二乘。优先用 scipy，没有就用纯 numpy 的投影梯度下降兜底。

    这个问题规模不大（路段数 × 路径数），而且加了强先验正则，两种解法结果接近。
    """
    try:
        from scipy.optimize import nnls as _scipy_nnls
        x, _ = _scipy_nnls(A, b)
        return x
    except Exception:
        pass
    # 投影梯度下降。L 用 Frobenius 范数的平方作谱范数上界，保证收敛。
    x = np.zeros(A.shape[1], dtype=np.float64)
    L = float((A * A).sum()) + 1e-12
    Atb = A.T @ b
    for _ in range(1200):
        x_new = np.maximum(0.0, x - (A.T @ (A @ x) - Atb) / L)
        delta = float(np.max(np.abs(x_new - x)))
        x = x_new
        if delta < 1e-8 * max(1.0, float(np.max(np.abs(x)))):
            break
    return x


def build_odme_predictions(root, panel, split, cfg):
    net = root / "corridors" / panel / "network"
    tdir = root / "task4" / panel / split
    paths_p = net / "path_set.csv"
    inc_p = net / "path_link_incidence.csv"
    cnt_p = tdir / "synthetic_link_counts.csv"
    pri_p = tdir / "synthetic_weak_prior.csv"
    if not (paths_p.exists() and inc_p.exists()):
        return None

    paths = pd.read_csv(paths_p, dtype={"path_id": str})
    if "path_id" not in paths.columns:
        return None
    paths["path_id"] = paths.path_id.astype(str)
    path_ids = paths["path_id"].tolist()

    inc = pd.read_csv(inc_p, dtype={"path_id": str, "link_id": str})
    if not {"path_id", "link_id"}.issubset(inc.columns):
        return None
    inc["path_id"] = inc.path_id.astype(str)
    inc["link_id"] = inc.link_id.astype(str)

    pidx = {p: i for i, p in enumerate(path_ids)}
    A = np.zeros((inc.link_id.nunique(), len(path_ids)), dtype=np.float64)
    link_ids = inc.link_id.drop_duplicates().tolist()
    lidx = {l: i for i, l in enumerate(link_ids)}
    rr = inc.link_id.map(lidx).to_numpy()
    cc = inc.path_id.map(pidx).to_numpy()
    ok = pd.notna(rr) & pd.notna(cc)
    A[rr[ok].astype(int), cc[ok].astype(int)] = 1.0

    # 先验 b
    b = np.zeros(len(path_ids))
    departure_time = "PUBLIC"
    if pri_p.exists():
        pri = pd.read_csv(pri_p, dtype={"path_id": str})
        pri["path_id"] = pri.path_id.astype(str)
        if "departure_time" in pri.columns and len(pri):
            departure_time = str(pri["departure_time"].iloc[0])
        if "path_flow" in pri.columns:
            s = pri.drop_duplicates("path_id").set_index("path_id")["path_flow"]
            s = pd.to_numeric(s, errors="coerce")
            b = s.reindex(path_ids).fillna(0.0).to_numpy(dtype=float)

    # 观测计数 c
    if cnt_p.exists():
        cnt = pd.read_csv(cnt_p, dtype={"link_id": str})
        cnt["link_id"] = cnt.link_id.astype(str)
        col = "count" if "count" in cnt.columns else cnt.columns[-1]
        c_full = pd.to_numeric(cnt.set_index("link_id")[col], errors="coerce")
        c_all = c_full.reindex(link_ids).fillna(0.0).to_numpy(dtype=float)
        measured = c_full.reindex(link_ids).notna().to_numpy()
    else:
        # 没有发布计数时，用 train 的晚高峰均值近似（兜底，效果会差）
        c_all = np.zeros(len(link_ids))
        measured = np.zeros(len(link_ids), dtype=bool)

    lam = float(cfg["odme_lambda"])
    if measured.sum() > 0:
        Am = A[measured]
        cm = c_all[measured]
        aa = np.vstack([Am, np.sqrt(lam) * np.eye(A.shape[1])])
        bb = np.concatenate([cm, np.sqrt(lam) * b])
        f = _nnls(aa, bb)
    else:
        f = b.copy()

    out = pd.DataFrame({
        "panel": panel,
        "departure_time": departure_time,
        "path_id": path_ids,
        "path_flow": f,
    })
    for c in ("origin_zone", "destination_zone"):
        out[c] = paths[c].astype(str).values if c in paths.columns else ""
    return out[["panel", "departure_time", "path_id", "origin_zone", "destination_zone", "path_flow"]]


In [ ]:
# ============================================================
# 8. 主流程
# ============================================================

cfg_file = ROOT / "config" / "corridors.json"
if cfg_file.exists():
    manifest = json.loads(cfg_file.read_text(encoding="utf-8"))
    panels = [p["corridor_id"] for p in manifest.get("panels", [])]
else:
    panels = sorted(p.name for p in (ROOT / "corridors").iterdir() if p.is_dir())
if CFG["panels"]:
    panels = [p for p in panels if p in set(CFG["panels"])]
log(f"要处理的走廊（{len(panels)} 条）:", panels)

state_parts, queue_parts, odme_parts = [], [], []

for panel in panels:
    panel_dir = ROOT / "corridors" / panel
    if not panel_dir.is_dir():
        log(f"[跳过] {panel}: 目录不存在")
        continue
    log(f"===== {panel} =====")
    net = network_params(panel_dir)
    milepost = link_mileposts(panel_dir)

    prof_s, prof_f, prof_links = build_profiles(panel_dir, CFG)

    for split in CFG["splits"]:
        if not (panel_dir / split).is_dir():
            continue
        log(f"  -- {panel} / {split} --")

        # Task 1
        try:
            pred = predict_split(panel_dir, split, net, prof_s, prof_f, prof_links, milepost, CFG)
            st = apply_state_predictions(ROOT, panel, split, pred, CFG)
            if st is not None and len(st):
                state_parts.append(st)
                log(f"    Task1 行数 {len(st):,}")
        except Exception as e:
            log(f"    [Task1 失败] {type(e).__name__}: {e}")

        # Task 2
        try:
            q = build_queue_predictions(ROOT, panel, split, net, CFG)
            if q is not None and len(q):
                queue_parts.append(q)
                log(f"    Task2 行数 {len(q):,}")
        except Exception as e:
            log(f"    [Task2 失败] {type(e).__name__}: {e}")

        # Task 4
        try:
            o = build_odme_predictions(ROOT, panel, split, CFG)
            if o is not None and len(o):
                odme_parts.append(o)
                log(f"    Task4 行数 {len(o):,}")
        except Exception as e:
            log(f"    [Task4 失败] {type(e).__name__}: {e}")

    del prof_s, prof_f
    gc.collect()

state = pd.concat(state_parts, ignore_index=True) if state_parts else pd.DataFrame(
    columns=["panel", "timestamp", "station_id", "link_id", "mask_regime", "speed_kmh", "flow_vph"])
queue = pd.concat(queue_parts, ignore_index=True) if queue_parts else pd.DataFrame(
    columns=["window_id", "timestamp", "link_id", "queue_pred"])
odme = pd.concat(odme_parts, ignore_index=True) if odme_parts else pd.DataFrame(
    columns=["panel", "departure_time", "path_id", "origin_zone", "destination_zone", "path_flow"])

log(f"汇总：state={len(state):,}  queue={len(queue):,}  odme={len(odme):,}")


In [ ]:
# ============================================================
# 9. Task 2 线下评估 + 机器学习版排队预测
# ============================================================
# 官方把队列标签全扣了。但标签的定义是公开的：
#   queued  ⇔  speed_kmh <= v_cut（v_cut = 0.60 × free_speed）
# 而且官方说真值取自「底层状态」——train 的 mainline_states 就是那个底层状态
# （它同时也是 Task 1 的答案层）。所以 train 上可以自己造出队列标签。
#
# 窗口规则照抄 config/synthetic_release_v1.json：
#   历史 60 分钟（12 步）→ 预测 T+5..T+30（6 步）
#   queue_ongoing：T 时已有路段排队；queue_onset：T 时无排队但视野内会出现
#   视野内必须有排队；窗口起点间隔 >= 360 分钟；历史覆盖率 >= 0.7
#
# 特征的关键约束：推理时官方只给 window_history.parquet（12 步速度），
# 所以【训练和推理必须共用同一个特征函数】。而 network/ 在推理时也能读到，
# 于是拓扑、车道数、v_cut、历史排队频率都可以用。

QUEUE_CFG = dict(
    panels=None,              # None = 全部走廊；建议先 ["D12_I5_N"]
    max_train_days=0,         # 0 = 全部 train 日文件
    holdout_frac=0.25,        # 时间轴后 25% 只用于【评估】，前 75% 只用于【训练】
    windows_per_cond=400,     # 每种 condition 最多取多少窗口训练（两种合计 2 倍）
    min_spacing_train=12,     # 训练窗口起点最小间隔（步；12 = 1 小时），降低自相关
    windows_per_panel=800,    # 每条走廊训练窗口总数上限
    links_per_window=90,      # 每个窗口最多取多少路段（含全部"有风险"的）
    min_spacing_min=360,      # 官方：评估窗口起点最小间隔
    eval_per_cond=15,         # 自建评估窗口数。官方是 5，但 5 个窗口的标准误高达 0.05，
                              # 调阈值/比小改动根本分辨不了（v4~v7 的教训）。
                              # 离线诊断用 15，噪声降到 ~0.03。
    min_hist_coverage=0.7,    # 官方：历史最小覆盖率
    max_persistence_iou=0.9,  # 官方：持续性 IoU 上限，太简单的窗口剔除
)

HIST_STEPS = 12
FUT_STEPS = 6
PEAK_HOURS = (6, 7, 8, 9, 16, 17, 18, 19)


def build_state_matrix(panel_dir, split, max_files=0):
    """把一个 split 的观测聚成三张同形状矩阵：(速度, 合格掩码, 流量)。

    返回 (SP, OK, FP)：index 都是 timestamp，columns 都是 link_id。
    FP 是这一版新加的 —— Task 2 的窗口历史只给速度，
    但窗口历史的流量其实也在发布包里（见 features_from_history 的说明）。
    """
    files = sorted((panel_dir / split / "mainline_states_masked").glob("**/*.parquet"))
    if not files:
        files = sorted((panel_dir / split / "mainline_states").glob("**/*.parquet"))
    if not files:
        return None
    if max_files and len(files) > max_files:
        idx = np.linspace(0, len(files) - 1, max_files).astype(int)
        files = [files[i] for i in sorted(set(idx))]
    parts = []
    for p in files:
        d = read_parquet_smart(p, ["timestamp", "link_id", "speed_kmh", "flow_vph",
                                   "is_score_eligible", "pct_observed"])
        if not len(d) or "link_id" not in d.columns:
            continue
        d = as_link_str(as_ts(d))
        if "is_score_eligible" in d.columns:
            d["_ok"] = d["is_score_eligible"].astype(bool)
        elif "pct_observed" in d.columns:
            d["_ok"] = pd.to_numeric(d["pct_observed"], errors="coerce") >= 75
        else:
            d["_ok"] = True
        for c in ("speed_kmh", "flow_vph"):
            if c not in d.columns:
                d[c] = np.nan
        parts.append(d[["timestamp", "link_id", "speed_kmh", "flow_vph", "_ok"]])
    if not parts:
        return None
    obs = pd.concat(parts, ignore_index=True)
    g = obs.groupby(["timestamp", "link_id"], as_index=False).agg(
        sp=("speed_kmh", "mean"), fl=("flow_vph", "mean"), ok=("_ok", "max"))
    SP = g.pivot(index="timestamp", columns="link_id", values="sp").sort_index()
    OK = g.pivot(index="timestamp", columns="link_id", values="ok").sort_index()
    FP = g.pivot(index="timestamp", columns="link_id", values="fl").sort_index()
    cols = SP.columns
    OK = OK.reindex(index=SP.index, columns=cols).fillna(False).astype(bool)
    FP = FP.reindex(index=SP.index, columns=cols)
    return SP, OK, FP


def link_topology(panel_dir):
    """link_id -> (上游列表, 下游列表)。拥堵向上游传播，所以下游排队是前兆。"""
    p = panel_dir / "network" / "lwr_mainline_topology.csv"
    up, down = {}, {}
    if not p.exists():
        return up, down
    t = pd.read_csv(p, dtype=str)
    if "link_id" not in t.columns:
        return up, down
    for r in t.itertuples(index=False):
        lid = str(getattr(r, "link_id"))
        inc = str(getattr(r, "incoming_link_ids", "") or "")
        out = str(getattr(r, "outgoing_link_ids", "") or "")
        up[lid] = [x for x in inc.split(";") if x]
        down[lid] = [x for x in out.split(";") if x]
    return up, down


def queue_context(panel_dir, net, cols, SPt, OKt):
    """构造特征需要的静态上下文：v_cut、车道数、自由流、拓扑、历史排队频率。

    历史排队频率是从 train 统计出来的表，推理时直接查表。
    SPt / OKt 是已经建好的 train 状态矩阵 —— 由调用方传进来，
    避免为了统计频率把 train 再读一遍（那是整整一倍的读取时间）。
    """
    neti = net.set_index("link_id").reindex(cols)

    def _col(name, default):
        if name in neti.columns:
            return pd.to_numeric(neti[name], errors="coerce").fillna(default)
        return pd.Series(default, index=neti.index, dtype=float)

    vcut = _col("v_cut", 63.0).to_numpy(float)
    lanes = _col("lanes", 4.0).clip(lower=1.0).to_numpy(float)
    vf = _col("free_speed_kmh", 105.0).to_numpy(float)
    ln = _col("length_km", 0.8).to_numpy(float)
    cap = _col("capacity_vph", 0.0).to_numpy(float)
    cap = np.where(cap > 0, cap, 1800.0 * lanes)      # 缺通行能力就按 1800/车道 兜底
    kcrit = cap / np.maximum(vf, 1.0)                  # 临界密度 = capacity / free_speed
    up, down = link_topology(panel_dir)
    pos = {c: i for i, c in enumerate(cols)}
    up_idx = [np.array([pos[x] for x in up.get(c, []) if x in pos], dtype=int) for c in cols]
    down_idx = [np.array([pos[x] for x in down.get(c, []) if x in pos], dtype=int) for c in cols]

    # 历史排队频率（路段 × 小时桶）+ 瓶颈排名
    freq = np.full((len(cols), 24), np.nan)
    bottle = np.zeros(len(cols))
    # 走廊级身份特征：各走廊的排队基础率差 8 倍（0.025 ~ 0.209），
    # 而原来的特征里完全没有「我在哪条路上」这个信息。
    panel_qrate = np.nan
    panel_v_ratio = np.nan
    if SPt is not None and OKt is not None:
        St = SPt.to_numpy(float)
        Vt = OKt.to_numpy() & np.isfinite(St)
        colpos = {c: i for i, c in enumerate(SPt.columns)}
        keep = np.array([colpos.get(c, -1) for c in cols])
        netcol = net.set_index("link_id").reindex(SPt.columns)
        if "v_cut" in netcol.columns:
            vc_t = pd.to_numeric(netcol["v_cut"], errors="coerce").fillna(63.0).to_numpy(float)
        else:
            vc_t = np.full(len(SPt.columns), 63.0)
        Qt = (St <= vc_t[None, :]) & Vt
        hours = SPt.index.hour.to_numpy()
        for h in range(24):
            m = hours == h
            if m.any():
                with np.errstate(invalid="ignore"):
                    fh = np.nanmean(np.where(Vt[m], Qt[m], np.nan), axis=0)
                for j, k in enumerate(keep):
                    if k >= 0:
                        freq[j, h] = fh[k]
        for j, k in enumerate(keep):
            if k >= 0:
                with np.errstate(invalid="ignore"):
                    bottle[j] = np.nanmean(St[:, k])
        # 走廊级汇总
        with np.errstate(invalid="ignore"):
            panel_qrate = float(np.nanmean(Qt.astype(float)))
            vf_t = (pd.to_numeric(netcol["free_speed_kmh"], errors="coerce").fillna(105.0).to_numpy(float)
                    if "free_speed_kmh" in netcol.columns
                    else np.full(len(SPt.columns), 105.0))
            panel_v_ratio = float(np.nanmean(St / np.maximum(vf_t, 1.0)[None, :]))
        del St, Vt, Qt
    # 排名归一到 0..1，越小越瓶颈
    ok = np.isfinite(bottle) & (bottle > 0)
    if ok.any():
        r = np.full(len(cols), 0.5)
        order = np.argsort(np.argsort(bottle[ok]))
        r[ok] = order / max(len(order) - 1, 1)
        bottle = r
    freq = np.where(np.isfinite(freq), freq, np.nan)
    return dict(vcut=vcut, lanes=lanes, vf=vf, ln=ln, cap=cap, kcrit=kcrit,
                freq=freq, bottle=bottle, up=up_idx, down=down_idx,
                panel_qrate=panel_qrate, panel_v_ratio=panel_v_ratio)


LINK_FEATURES = ["vcut", "lanes", "free_speed", "length_km",
                 "s_last", "ratio_last", "s_mean", "s_min", "s_std", "slope",
                 "ratio_mean", "ratio_min", "n_queued", "queued_at_T",
                 "down_queued", "down_min_ratio", "up_queued",
                 "hist_freq", "bottleneck", "hour", "weekday", "is_peak", "is_onset",
                 # ↓ 流量 / 密度。官方对 onset 可解性的说明点名的两个判别量是
                 #   「速度比值」和「流量」（1.28x / 6912 vph vs 1.54x / 4620 vph）。
                 #   速度比值原来就有，流量完全没有 —— 这一版补上。
                 "f_last", "f_mean", "f_std", "f_slope",
                 "k_last", "k_over_kcrit", "q_over_cap",
                 # ↓ 走廊身份。各走廊排队基础率差 8 倍（0.025 ~ 0.209），
                 #   而原特征里没有任何「我在哪条路上」的信息。
                 "panel_qrate", "panel_v_ratio"]

# 视野步序号必须单独作为一列特征。
# onset 窗口的排队往往出现在第 4-6 步；如果模型对 6 个步只能给出同一个判断，
# 这个时间结构就白白丢掉了 —— 这是上一版模型退步的原因之一。
FEATURE_NAMES = LINK_FEATURES + ["horizon_step"]


def tile_with_steps(Xlink):
    """(n_links, 23) -> (n_links*FUT_STEPS, 24)，行序为「路段优先」。

    路段优先是为了能直接 reshape(n_links, FUT_STEPS) 还原成矩阵。
    """
    n = len(Xlink)
    Xi = np.repeat(Xlink.to_numpy(float), FUT_STEPS, axis=0)
    step = np.tile(np.arange(FUT_STEPS, dtype=float), n)
    return pd.DataFrame(np.column_stack([Xi, step]), columns=FEATURE_NAMES)


def features_from_history(HS, HK, ctx, hour, weekday, is_onset, HF=None):
    """训练与推理共用的特征函数。

    HS : (HIST_STEPS, n_links) 速度历史，最后一行是 T
    HK : (HIST_STEPS, n_links) 合格掩码
    HF : (HIST_STEPS, n_links) 流量历史；None 时流量类特征为 NaN
    返回 (n_links, len(LINK_FEATURES))

    HF 从哪来：官方只把窗口历史的**速度**放进了 window_history.parquet，
    但窗口的 horizon 及其后一小时才被从主线层移除，**历史本身是 republished 的**
    （所以它被排除在 Task 1 掩码之外）。也就是说这些时间戳的流量就在
    mainline_states_masked 里，只是之前没去读。官方对 onset 的判别量
    正是「速度比值 + 流量」，而流量一直缺着。
    """
    n = HS.shape[1]
    vc = ctx["vcut"]
    V = HK & np.isfinite(HS)
    Q = (HS <= vc[None, :]) & V
    with np.errstate(invalid="ignore", divide="ignore"):
        s_last = HS[-1]
        ratio_last = s_last / vc
        s_mean = np.nanmean(np.where(V, HS, np.nan), axis=0)
        s_min = np.nanmin(np.where(V, HS, np.nan), axis=0)
        s_std = np.nanstd(np.where(V, HS, np.nan), axis=0)
        # 线性斜率：t = 0..H-1，最后一行是 T
        t = np.arange(HIST_STEPS, dtype=float)
        tb = t - t.mean()
        denom = float((tb * tb).sum())
        Sv = np.where(V, HS, np.nan)
        sbar = np.nanmean(Sv, axis=0)
        slope = np.nansum((Sv - sbar[None, :]) * tb[:, None], axis=0) / max(denom, 1e-9)
        ratio_mean = s_mean / vc
        ratio_min = s_min / vc
        n_queued = Q.sum(axis=0).astype(float)
        queued_T = Q[-1].astype(float)

        # --- 流量 / 密度 ---
        if HF is not None and HF.shape[1] == n:
            Hv = np.where(V & np.isfinite(HF), HF, np.nan)
            f_last = HF[-1]
            f_mean = np.nanmean(Hv, axis=0)
            f_std = np.nanstd(Hv, axis=0)
            f_bar = np.nanmean(Hv, axis=0)
            f_slope = np.nansum((Hv - f_bar[None, :]) * tb[:, None], axis=0) / max(denom, 1e-9)
        else:
            nanv = np.full(n, np.nan)
            f_last = f_mean = f_std = f_slope = nanv
        k_last = f_last / np.maximum(s_last, 1.0)                  # 密度 k = q / v
        k_over_kcrit = k_last / np.maximum(ctx["kcrit"], 1e-6)     # 离拥堵的物理距离
        q_over_cap = f_last / np.maximum(ctx["cap"], 1e-6)         # 通行能力占用率

        # 下游（本路段汇入的方向）排队 → 拥堵向上游传播的前兆
        down_queued = np.zeros(n)
        down_min_ratio = np.full(n, np.nan)
        for j in range(n):
            ix = ctx["down"][j]
            if ix.size:
                down_queued[j] = float(queued_T[ix].mean())
                rr = ratio_last[ix]
                rr = rr[np.isfinite(rr)]
                if rr.size:
                    down_min_ratio[j] = float(rr.min())
        up_queued = np.zeros(n)
        for j in range(n):
            ix = ctx["up"][j]
            if ix.size:
                up_queued[j] = float(queued_T[ix].mean())

    hb = int(hour) % 24
    hist_freq = ctx["freq"][:, hb]
    X = np.column_stack([
        vc, ctx["lanes"], ctx["vf"], ctx["ln"],
        s_last, ratio_last, s_mean, s_min, s_std, slope,
        ratio_mean, ratio_min, n_queued, queued_T,
        down_queued, down_min_ratio, up_queued,
        hist_freq, ctx["bottle"],
        np.full(n, hour, dtype=float), np.full(n, weekday, dtype=float),
        np.full(n, 1.0 if int(hour) in PEAK_HOURS else 0.0),
        np.full(n, 1.0 if is_onset else 0.0),
        f_last, f_mean, f_std, f_slope,
        k_last, k_over_kcrit, q_over_cap,
        np.full(n, ctx.get("panel_qrate", np.nan), dtype=float),
        np.full(n, ctx.get("panel_v_ratio", np.nan), dtype=float),
    ])
    return pd.DataFrame(X, columns=LINK_FEATURES)


def effective_coverage(V, cfg):
    """自建窗口的历史覆盖率下限。

    官方的 minimum_history_coverage=0.7 是针对 window_history.parquet 那份
    【专门发布的、未掩码的】Task 2 历史。而我们是从被 Task 1 掩码过的主线层
    自建窗口，观测密度天然只有 V.mean() 那么多。直接套 0.7 有可能把
    所有窗口都过滤掉，导致评估表根本打印不出来。

    所以取「不超过数据本身能提供的密度」：min(0.7, 0.8 × V.mean())。
    这只影响【自建窗口的自我评估】，不影响提交本身。
    """
    dens = float(V.mean()) if V.size else 0.0
    return float(min(cfg["min_hist_coverage"], 0.8 * dens))


def window_condition(Q, V, i0, min_cov=None):
    """判断窗口属于哪种 condition，并检查官方规则。

    注意 min_hist_coverage 指的是【历史里有观测的比例】，不是排队的比例 ——
    用排队比例来卡会让所有窗口都被过滤掉（排队率远低于 0.7）。
    """
    if min_cov is None:
        min_cov = QUEUE_CFG["min_hist_coverage"]
    hist = Q[max(i0 - HIST_STEPS + 1, 0): i0 + 1]
    vhist = V[max(i0 - HIST_STEPS + 1, 0): i0 + 1]
    fut = Q[i0 + 1: i0 + 1 + FUT_STEPS]
    if fut.size == 0 or not fut.any():
        return None                      # 视野内必须有排队
    cov = float(vhist.mean()) if vhist.size else 0.0
    if cov < min_cov:
        return None
    return "queue_ongoing" if hist[-1].any() else "queue_onset"


def select_eval_windows(SP, OK, cols, ctx, per_cond=None):
    """按官方规则挑评估窗口：每种 condition 每走廊 N 个，起点间隔 >= 360 分钟。"""
    per_cond = per_cond or QUEUE_CFG["eval_per_cond"]
    St = SP.to_numpy(float)
    V = OK.to_numpy() & np.isfinite(St)
    Q = (St <= ctx["vcut"][None, :]) & V
    idx = SP.index
    n = len(idx)
    if n < HIST_STEPS + FUT_STEPS + 2:
        return {}
    spacing = QUEUE_CFG["min_spacing_min"] // 5
    min_cov = effective_coverage(V, QUEUE_CFG)
    picked = {"queue_onset": [], "queue_ongoing": []}
    # ---- 时间切分（防泄漏）----
    # 评估窗口只从时间轴的【后段】挑，训练窗口只用【前段】，两者绝不重叠。
    # 否则评估窗口本身就是训练样本，模型分数会严重虚高 ——
    # 这正是上一版出现 model 0.31 vs rule 0.11 这种反常差距的原因。
    lo = int(n * (1.0 - QUEUE_CFG["holdout_frac"]))
    for i0 in range(max(HIST_STEPS - 1, lo), n - FUT_STEPS):
        cond = window_condition(Q, V, i0, min_cov)
        if cond is None or len(picked[cond]) >= per_cond:
            continue
        if picked[cond] and (i0 - picked[cond][-1]) < spacing:
            continue
        # 剔除持续性就能拿高分的太简单窗口
        pers = Q[i0][None, :].repeat(FUT_STEPS, axis=0)
        fut = Q[i0 + 1: i0 + 1 + FUT_STEPS]
        inter = np.logical_and(pers, fut).sum()
        union = np.logical_or(pers, fut).sum()
        iou = 1.0 if union == 0 else inter / union
        if iou >= QUEUE_CFG["max_persistence_iou"]:
            continue
        picked[cond].append(i0)
        if all(len(v) >= per_cond for v in picked.values()):
            break
    return picked


def iou_of(pred, truth):
    inter = np.logical_and(pred, truth).sum()
    union = np.logical_or(pred, truth).sum()
    return 1.0 if union == 0 else float(inter / union)


def evaluate_windows(SP, OK, cols, ctx, picked, predictor, threshold=0.5, FP=None):
    """在自建窗口上评估一个预测器。

    predictor(Xlink, meta) 必须返回 (n_links, FUT_STEPS) 的概率矩阵 ——
    逐步预测，而不是对 6 个步给同一个判断。
    """
    St = SP.to_numpy(float)
    V = OK.to_numpy() & np.isfinite(St)
    Q = (St <= ctx["vcut"][None, :]) & V
    Ft = FP.to_numpy(float) if FP is not None else None
    idx = SP.index
    rows = []
    for cond, positions in picked.items():
        for i0 in positions:
            sl = slice(i0 - HIST_STEPS + 1, i0 + 1)
            HS = St[sl]
            HK = V[sl]
            HF = Ft[sl] if Ft is not None else None
            hour = idx[i0].hour
            wd = idx[i0].weekday()
            Xlink = features_from_history(HS, HK, ctx, hour, wd,
                                          cond == "queue_onset", HF)
            # Q 的行是时间、列是路段；预测是 (路段, 步)，所以真值要转置对齐
            truth = Q[i0 + 1: i0 + 1 + FUT_STEPS].T          # (n_links, FUT_STEPS)
            prob = np.asarray(predictor(Xlink, dict(cond=cond, i0=i0)), dtype=float)
            prob = prob.reshape(len(cols), FUT_STEPS)
            pred = prob > threshold
            rows.append(dict(condition=cond, i0=i0, timestamp=str(idx[i0]),
                             iou=iou_of(pred, truth), n_true=int(truth.sum())))
    return pd.DataFrame(rows)


def build_training_data(panel_dir, SP, OK, cols, ctx, cfg, rng, FP=None):
    """从 train 造训练样本 —— 关键是【用与官方窗口同样的接受条件】挑起点。

    上一版这里每天随机取 10 个起点，造出来的窗口绝大多数根本不堵；
    而官方的评估窗口【保证视野内出现排队】。训练集学成"基本不堵"，
    测试集却全是堵的 —— 典型的协变量偏移，模型因此在榜上退步。
    现在改成和推理完全同一套接受条件，并按 condition 均衡采样。
    """
    St = SP.to_numpy(float)
    V = OK.to_numpy() & np.isfinite(St)
    Q = (St <= ctx["vcut"][None, :]) & V
    idx = SP.index
    n = len(idx)
    if n < HIST_STEPS + FUT_STEPS + 2:
        return None, None
    Ft = FP.to_numpy(float) if FP is not None else None

    cap = cfg["windows_per_cond"]
    spacing = max(int(cfg["min_spacing_train"]), 1)
    min_cov = effective_coverage(V, cfg)
    accepted = {"queue_onset": [], "queue_ongoing": []}
    last = {"queue_onset": -10 ** 9, "queue_ongoing": -10 ** 9}
    # ---- 时间切分（防泄漏）----
    # 只用时间轴【前段】造训练窗口，后段完整留给评估。
    # 不减 FUT_STEPS 也没关系，但减掉更干净：保证训练窗口的视野不越过切分点。
    hi = int(n * (1.0 - cfg["holdout_frac"])) - FUT_STEPS
    for i0 in range(HIST_STEPS - 1, max(hi, HIST_STEPS)):
        cond = window_condition(Q, V, i0, min_cov)
        if cond is None or len(accepted[cond]) >= cap:
            continue
        if (i0 - last[cond]) < spacing:
            continue
        # 与官方一致：剔除持续性就能拿高分的太简单窗口
        pers = Q[i0][None, :].repeat(FUT_STEPS, axis=0)
        fut = Q[i0 + 1: i0 + 1 + FUT_STEPS]
        inter = np.logical_and(pers, fut).sum()
        union = np.logical_or(pers, fut).sum()
        if union > 0 and (inter / union) >= cfg["max_persistence_iou"]:
            continue
        accepted[cond].append(i0)
        last[cond] = i0
        if all(len(v) >= cap for v in accepted.values()):
            break

    pick = [(i0, c) for c, v in accepted.items() for i0 in v]
    if not pick:
        return None, None
    if len(pick) > cfg["windows_per_panel"]:
        sel = rng.choice(len(pick), size=cfg["windows_per_panel"], replace=False)
        pick = [pick[i] for i in sel]

    Xs, ys = [], []
    vc = ctx["vcut"]
    for i0, cond in pick:
        sl = slice(i0 - HIST_STEPS + 1, i0 + 1)
        HS = St[sl]
        HK = V[sl]
        HF = Ft[sl] if Ft is not None else None
        if float(HK.mean()) < min_cov:
            continue
        Xlink = features_from_history(HS, HK, ctx, idx[i0].hour, idx[i0].weekday(),
                                      cond == "queue_onset", HF)
        fut = Q[i0 + 1: i0 + 1 + FUT_STEPS]            # (6, n_links)
        # 选路段：有风险的（历史或视野内排过队，或比值贴近阈值）全保留，再随机补足
        risky = Q[i0 - HIST_STEPS + 1: i0 + 1].any(axis=0) | fut.any(axis=0)
        with np.errstate(invalid="ignore"):
            near = np.nan_to_num(HS[-1] / vc, nan=9.9) < 1.5
        risk = risky | near
        ridx = np.flatnonzero(risk)
        oidx = np.flatnonzero(~risk)
        room = max(cfg["links_per_window"] - len(ridx), 0)
        if room and len(oidx):
            oidx = rng.choice(oidx, size=min(room, len(oidx)), replace=False)
        else:
            oidx = np.array([], dtype=int)
        sel = np.concatenate([ridx, oidx]).astype(int)
        if not len(sel):
            continue
        # 展开成「路段 × 视野步」，并带上 horizon_step 特征
        Xs.append(tile_with_steps(Xlink.iloc[sel]).to_numpy(float))     # (n_sel*6, 24)
        ys.append(fut[:, sel].T.reshape(-1).astype(np.int8))            # (n_sel*6,)
    if not Xs:
        return None, None
    return np.concatenate(Xs, axis=0), np.concatenate(ys, axis=0)


def _fit_model(X, y):
    """优先 LightGBM；没有就退到 sklearn 的 HistGradientBoosting；再不行用常数。"""
    try:
        import lightgbm as lgb
        m = lgb.LGBMClassifier(
            n_estimators=400, learning_rate=0.05, num_leaves=63,
            min_child_samples=40, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, reg_lambda=1.0, random_state=0,
            n_jobs=-1, verbose=-1)
        m.fit(X, y, feature_name=FEATURE_NAMES)
        return m, "lightgbm"
    except Exception as e:
        log(f"      LightGBM 不可用（{type(e).__name__}），退到 sklearn")
    try:
        from sklearn.ensemble import HistGradientBoostingClassifier
        m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06,
                                           max_leaf_nodes=63, random_state=0)
        m.fit(X, y)
        return m, "sklearn_hgb"
    except Exception as e:
        log(f"      sklearn 也不可用（{type(e).__name__}），退到常数先验")
        p = float(np.mean(y)) if len(y) else 0.0
        return (lambda Z: np.full(len(Z), p)), "constant"


# ---- 主流程：逐走廊造数据、训练、评估、预测 ----
q_panels = QUEUE_CFG["panels"] or panels
q_rng = np.random.default_rng(20261013)
q_train_X, q_train_y = [], []
q_ctxs, q_picks, q_evals = {}, {}, []
q_model = None
q_threshold = 0.5

for panel in q_panels:
    panel_dir = ROOT / "corridors" / panel
    if not panel_dir.is_dir():
        continue
    net_p = network_params(panel_dir)
    sm_tr = build_state_matrix(panel_dir, "train", QUEUE_CFG["max_train_days"])
    if sm_tr is None:
        log(f"  [Task2] {panel}: train 无观测，跳过")
        continue
    SP_tr, OK_tr, FP_tr = sm_tr
    cols = list(SP_tr.columns)
    # 特征统计一律用【全部 train】。
    # 关键：hist_freq 是模型输入，训练和推理必须同分布。
    # 从 75% 的 train 估出来和从 100% 估出来，含义相同但噪声水平不同；
    # 训练用一种、推理用另一种会引入隐性偏差（v6 实测掉到 0.75128）。
    # 代价：离线评估里这个特征略乐观（评估段的信息也进了统计）。
    # 记住这点 —— 离线数字只用来看【相对高低】，不要当绝对水平。
    ctx = queue_context(panel_dir, net_p, cols, SP_tr, OK_tr)
    q_ctxs[panel] = (cols, ctx, SP_tr, OK_tr, FP_tr, net_p)

    X, y = build_training_data(panel_dir, SP_tr, OK_tr, cols, ctx, QUEUE_CFG, q_rng, FP_tr)
    if X is not None:
        q_train_X.append(X)
        q_train_y.append(y)
        log(f"  [Task2] {panel}: 训练样本 {len(X):,}  正例率 {y.mean():.4f}")
    else:
        log(f"  [Task2] {panel}: 没造出符合官方条件的窗口")

if not q_train_X:
    log("[Task2] 没有造出训练样本，保留规则版预测")
else:
    Xtr = np.vstack(q_train_X)
    ytr = np.concatenate(q_train_y)
    log(f"[Task2] 训练集 {Xtr.shape[0]:,} 行 × {Xtr.shape[1]} 特征，正例率 {ytr.mean():.4f}")
    q_model, q_kind = _fit_model(Xtr, ytr)
    log(f"[Task2] 模型类型：{q_kind}")

    # 三个预测器，统一返回 (n_links, FUT_STEPS) 概率矩阵
    def _persistence(Xlink, meta):
        p = Xlink["queued_at_T"].to_numpy(float)[:, None]
        return np.repeat(p, FUT_STEPS, axis=1)

    def _rule(Xlink, meta):
        r = np.nan_to_num(Xlink["ratio_last"].to_numpy(float), nan=9.9)
        return np.repeat(np.where(r <= 1.15, 1.0, 0.0)[:, None], FUT_STEPS, axis=1)

    def _model_prob(Xlink, meta):
        Z = tile_with_steps(Xlink)[FEATURE_NAMES]
        try:
            p = np.asarray(q_model.predict_proba(Z)[:, 1], dtype=float)
        except Exception:
            p = np.asarray(q_model.predict(Z), dtype=float)
        return p.reshape(len(Xlink), FUT_STEPS)

    # 收集每个走廊的评估窗口
    for panel in list(q_ctxs):
        cols, ctx, SP_tr, OK_tr, FP_tr, net_p = q_ctxs[panel]
        picked = select_eval_windows(SP_tr, OK_tr, cols, ctx)
        if picked and sum(len(v) for v in picked.values()):
            q_picks[panel] = picked
            log(f"  [Task2 评估] {panel}: 窗口 "
                f"onset={len(picked.get('queue_onset', []))} "
                f"ongoing={len(picked.get('queue_ongoing', []))}")
        else:
            log(f"  [Task2 评估] {panel}: 没挑到符合官方规则的窗口")

    def _eval_all(predictor, threshold):
        outs = []
        for panel, picked in q_picks.items():
            cols, ctx, SP_tr, OK_tr, FP_tr, net_p = q_ctxs[panel]
            r = evaluate_windows(SP_tr, OK_tr, cols, ctx, picked, predictor,
                                 threshold=threshold, FP=FP_tr)
            if len(r):
                r["panel"] = panel
                outs.append(r)
        return pd.concat(outs, ignore_index=True) if outs else pd.DataFrame()

    for name, fn in (("persistence", _persistence), ("rule", _rule)):
        r = _eval_all(fn, 0.5)
        if len(r):
            r["method"] = name
            q_evals.append(r)

    r = _eval_all(_model_prob, 0.5)
    if len(r):
        r["method"] = "model@0.50"
        q_evals.append(r)

    # 阈值扫描 —— 分 condition 各自最优。
    # 官方口径是两种 condition 等权平均（S_queue = (IoU_onset + IoU_ongoing)/2），
    # 所以理应各自调阈值。用一个全局阈值必然牺牲一方：onset 窗口里排队的格子少，
    # 全局最优会倾向于把预测压掉，于是 onset 直接归零。
    q_thr = {"queue_onset": 0.5, "queue_ongoing": 0.5}
    for cond in ("queue_onset", "queue_ongoing"):
        best_t, best_v = 0.5, -1.0
        for thr in np.arange(0.05, 0.96, 0.05):
            rr = _eval_all(_model_prob, float(thr))
            if not len(rr):
                continue
            sub = rr[rr.condition == cond]
            if len(sub) and sub["iou"].mean() > best_v:
                best_v, best_t = float(sub["iou"].mean()), float(thr)
        q_thr[cond] = best_t
        log(f"[Task2] {cond} 最优阈值 {best_t:.2f}（该 condition IoU {best_v:.4f}）")
    q_threshold = float(np.mean(list(q_thr.values())))   # 仅用于展示

    def _eval_model_percond():
        """按「分 condition 阈值」评估模型。"""
        outs = []
        for panel, picked in q_picks.items():
            cols, ctx, SP_tr, OK_tr, FP_tr, net_p = q_ctxs[panel]
            for cond, thr in q_thr.items():
                pos = picked.get(cond, [])
                if not pos:
                    continue
                r = evaluate_windows(SP_tr, OK_tr, cols, ctx, {cond: pos},
                                     _model_prob, threshold=thr, FP=FP_tr)
                if len(r):
                    r["panel"] = panel
                    outs.append(r)
        return pd.concat(outs, ignore_index=True) if outs else pd.DataFrame()

    rr = _eval_model_percond()
    if len(rr):
        rr["method"] = "model@分condition"
        q_evals.append(rr)

    if q_evals:
        ev_all = pd.concat(q_evals, ignore_index=True)
        print("\n=== Task 2 线下评估（自建窗口，train split）===")
        print("按 condition 平均 IoU：")
        print(ev_all.pivot_table(index="method", columns="condition",
                                 values="iou", aggfunc="mean").to_string())
        print("\n等权平均（官方口径：两种 condition 等权）：")
        means = ev_all.groupby("method")["iou"].mean().sort_values(ascending=False)
        print(means.to_string())
        print("\n官方持续性基线在榜上是 0.2518，可做参照。")
        print("这是【自建窗口 + 自建标签】的近似评估，只用来看方法的相对高低，")
        print("不要当成榜上绝对分数。")

        # ---- 决策闸门 ----
        # 模型没有【明显】赢过规则版，就不要换掉它。
        # 上一次榜上从 0.66479 退到 0.62611，就是因为跳过了这道闸门。
        base = max(float(means.get("persistence", 0.0)), float(means.get("rule", 0.0)))
        best_model = float(means.get("model@分condition",
                                     means.get(f"model@{q_threshold:.2f}",
                                               means.get("model@0.50", 0.0))))
        if best_model <= base + 0.01:
            log(f"[Task2] 闸门：模型 IoU {best_model:.4f} 未明显超过规则版 {base:.4f}"
                f" → 【保留规则版预测】")
            q_model = None
        else:
            log(f"[Task2] 闸门：模型 IoU {best_model:.4f} 超过规则版 {base:.4f}"
                f" → 采用模型版（阈值 onset={q_thr['queue_onset']:.2f} / "
                f"ongoing={q_thr['queue_ongoing']:.2f}）")

# ---- 只有通过闸门，才用模型覆盖规则版；未覆盖的窗口沿用规则版 ----
if q_model is not None:
    queue_rule = queue.copy()
    new_rows = []
    # 每个 (走廊, split) 只建一次「掩码层流量矩阵」，给所有窗口查表用
    _fp_cache = {}

    def _split_fp(pdir, sp):
        k = (str(pdir), sp)
        if k not in _fp_cache:
            sm = build_state_matrix(pdir, sp)
            _fp_cache[k] = sm[2] if sm is not None else None
        return _fp_cache[k]
    for panel in q_panels:
        if panel not in q_ctxs:
            continue
        cols, ctx, SP_tr, OK_tr, FP_tr, net_p = q_ctxs[panel]
        colpos = {c: i for i, c in enumerate(cols)}
        panel_dir = ROOT / "corridors" / panel
        for split in CFG["splits"]:
            tdir = ROOT / "task2" / panel / split
            hp = tdir / "window_history.parquet"
            tp = tdir / "sample_submission_queue.csv"
            if not (hp.exists() and tp.exists()):
                continue
            FPsp = _split_fp(panel_dir, split)
            tmpl = pd.read_csv(tp, dtype=str)
            if not {"window_id", "timestamp", "link_id"}.issubset(tmpl.columns):
                continue
            tmpl = as_link_str(tmpl)
            tmpl["window_id"] = tmpl.window_id.astype(str)
            hist = read_parquet_smart(hp, ["window_id", "timestamp", "link_id",
                                           "speed_kmh", "is_score_eligible"])
            if not len(hist):
                continue
            hist = as_link_str(as_ts(hist))
            hist["window_id"] = hist.window_id.astype(str)
            hist["speed_kmh"] = pd.to_numeric(hist["speed_kmh"], errors="coerce")

            cond_by_window = {}
            wp = tdir / "window_index.csv"
            if wp.exists():
                w = pd.read_csv(wp, dtype=str)
                if {"window_id", "condition"}.issubset(w.columns):
                    cond_by_window = dict(zip(w.window_id.astype(str), w.condition.astype(str)))

            wanted = set(tmpl.window_id.astype(str))
            for wid, h in hist.groupby("window_id", sort=False):
                if wid not in wanted:
                    continue
                t = tmpl[tmpl.window_id == wid][["window_id", "timestamp", "link_id"]].copy()
                if not len(t):
                    continue
                # 时间戳升序 = 视野步序（T+5 … T+30）。步数不是 6 就不冒险，留给规则版
                ts_sorted = sorted(t["timestamp"].astype(str).unique())
                if len(ts_sorted) != FUT_STEPS:
                    continue
                # 历史对齐成 (HIST_STEPS, n_links)
                hv = h.pivot_table(index="timestamp", columns="link_id",
                                   values="speed_kmh", aggfunc="mean").sort_index()
                hk = h.pivot_table(index="timestamp", columns="link_id",
                                   values="is_score_eligible", aggfunc="max")
                hv = hv.reindex(columns=cols)
                hk = hk.reindex(index=hv.index, columns=cols).fillna(False).astype(bool)
                HS = hv.to_numpy(float)
                HK = hk.to_numpy()
                # 流量：从掩码层取【同一批】历史时间戳。官方只把速度放进了
                # window_history，但窗口历史的观测是 republished 的
                # （只有 horizon 及其后一小时被移除），所以流量拿得到。
                HF = None
                if FPsp is not None:
                    try:
                        HF = FPsp.reindex(index=hv.index, columns=cols).to_numpy(float)
                    except Exception:
                        HF = None
                if HS.shape[0] < 2:
                    continue
                if HS.shape[0] < HIST_STEPS:
                    pad = HIST_STEPS - HS.shape[0]
                    HS = np.vstack([np.repeat(HS[:1], pad, axis=0), HS])
                    HK = np.vstack([np.repeat(HK[:1], pad, axis=0), HK])
                    if HF is not None:
                        HF = np.vstack([np.repeat(HF[:1], pad, axis=0), HF])
                # 三张矩阵必须一起裁到 HIST_STEPS，否则行数对不齐会广播报错
                HS = HS[-HIST_STEPS:]
                HK = HK[-HIST_STEPS:]
                if HF is not None:
                    HF = HF[-HIST_STEPS:]
                last_ts = hv.index.max()
                is_onset = cond_by_window.get(wid, "queue_ongoing") == "queue_onset"
                # 按该窗口的 condition 取对应的最优阈值
                thr_here = q_thr["queue_onset" if is_onset else "queue_ongoing"]
                Xlink = features_from_history(HS, HK, ctx, last_ts.hour,
                                              last_ts.weekday(), is_onset, HF)
                prob = _model_prob(Xlink, dict(cond=cond_by_window.get(wid), i0=None))

                step_of = {ts: k for k, ts in enumerate(ts_sorted)}
                si = t["timestamp"].astype(str).map(step_of).to_numpy()
                li = t["link_id"].map(colpos).to_numpy()
                good = pd.notna(si) & pd.notna(li)
                t = t.loc[good].copy()
                si = si[good].astype(int)
                li = li[good].astype(int)
                t["queue_pred"] = (prob[li, si] > thr_here).astype(int)
                new_rows.append(t)

    if new_rows:
        queue_model = pd.concat(new_rows, ignore_index=True).drop_duplicates(
            ["window_id", "timestamp", "link_id"])
        # 未覆盖的窗口沿用规则版，绝不能留下空档（空档会被当成 0 计分）
        kk = ["window_id", "timestamp", "link_id"]
        mg = queue_rule[kk + ["queue_pred"]].merge(
            queue_model[kk + ["queue_pred"]], on=kk, how="left",
            suffixes=("_rule", "_model"))
        mg["queue_pred"] = mg["queue_pred_model"].fillna(mg["queue_pred_rule"])
        queue = mg[kk + ["queue_pred"]].copy()
        queue["queue_pred"] = queue["queue_pred"].astype(int)
        log(f"[Task2] 模型版预测 {len(queue_model):,} 行，"
            f"合并后共 {len(queue):,} 行（未覆盖的沿用规则版）")


In [ ]:
# ============================================================
# 10. 合并成上传文件
# ============================================================
# 上传格式固定 6 列：submission_id, task, speed_kmh, flow_vph, queue_pred, path_flow
# 用 submission_key.csv 把三个任务的结果 join 上去。

OUT_COLS = ["submission_id", "task", "speed_kmh", "flow_vph", "queue_pred", "path_flow"]
KEYS = {
    "state": ["panel", "timestamp", "station_id", "link_id", "mask_regime"],
    "queue": ["window_id", "timestamp", "link_id"],
    "odme": ["panel", "departure_time", "path_id"],
}
VALUES = {"state": ["speed_kmh", "flow_vph"], "queue": ["queue_pred"], "odme": ["path_flow"]}


def index_task(df, task):
    """把某个任务的结果整理成以自然键为索引的表，供上传文件 join。

    注意：timestamp 必须转成 datetime，绝不能转成字符串 ——
    映射表 chunk 里的 timestamp 是 datetime，两边类型不一致会导致一行都对不上，
    整个 Task 1 会被静默写成 0。
    """
    if not len(df):
        return None
    d = df.copy()
    for c in KEYS[task]:
        if c == "timestamp":
            d[c] = pd.to_datetime(d[c], utc=True, errors="coerce")
        else:
            d[c] = d[c].astype(str)
    d = d.dropna(subset=KEYS[task]).drop_duplicates(KEYS[task])
    return d.set_index(KEYS[task])[VALUES[task]]


tables = {"state": index_task(state, "state"),
          "queue": index_task(queue, "queue"),
          "odme": index_task(odme, "odme")}

out_path = CFG["out_path"]


def resolve_key_path():
    """上传文件必须靠映射表把 submission_id 对应到具体格子。

    首选 submission_key.csv；若缺，退而找同样带 submission_id/task 的 sample_submission.csv。
    """
    for name in ("submission_key.csv", "sample_submission.csv"):
        p = ROOT / name
        if p.exists():
            cols = [c.strip() for c in pd.read_csv(p, nrows=1, encoding="utf-8-sig").columns]
            if {"submission_id", "task"}.issubset(cols):
                return p
    raise FileNotFoundError(
        "找不到 submission_key.csv（也没有可用的 sample_submission.csv）。\n"
        "合并上传文件必须用它把 submission_id 映射到具体格子。\n"
        f"数据根目录 {ROOT} 下现有: {sorted(os.listdir(ROOT))}"
    )


key_path = resolve_key_path()
log(f"映射表: {key_path.name}")

written, gaps = 0, {t: 0 for t in tables}
first = True
for chunk in pd.read_csv(key_path, chunksize=1_000_000, dtype=str, encoding="utf-8-sig"):
    chunk.columns = [c.strip() for c in chunk.columns]
    chunk["timestamp"] = pd.to_datetime(chunk["timestamp"], utc=True, errors="coerce")
    chunk["task"] = chunk["task"].astype(str)
    out = pd.DataFrame({
        "submission_id": pd.to_numeric(chunk["submission_id"], errors="coerce").astype("int64"),
        "task": chunk["task"],
    })
    for c in ("speed_kmh", "flow_vph", "queue_pred", "path_flow"):
        out[c] = 0.0
    for task, table in tables.items():
        if table is None:
            continue
        rows = (out["task"] == task).to_numpy()
        if not rows.any():
            continue
        wanted = pd.MultiIndex.from_frame(chunk.loc[rows, KEYS[task]])
        found = table.reindex(wanted)
        for c in VALUES[task]:
            vals = found[c].to_numpy()
            gaps[task] += int(pd.isna(vals).sum())
            out.loc[rows, c] = pd.Series(vals).fillna(0.0).to_numpy()
    out[OUT_COLS].to_csv(out_path, mode="w" if first else "a", header=first, index=False)
    first = False
    written += len(out)
    if written % 2_000_000 < 1_000_000:
        log(f"  已写 {written:,} 行")

log(f"写出 {written:,} 行 → {out_path}")
for t, n in gaps.items():
    if n:
        log(f"  [警告] {t} 有 {n:,} 行没匹配上，已写 0（会拉低分数）")
if not any(gaps.values()):
    log("所有计分格都已被你的结果填满")


In [ ]:
# ============================================================
# 11. 自检
# ============================================================
sub = pd.read_csv(out_path, nrows=5)
log("文件头：")
print(sub.to_string(index=False))

full = pd.read_csv(out_path, usecols=["task"] + ["speed_kmh", "flow_vph", "queue_pred", "path_flow"])
log("各任务行数：")
print(full["task"].value_counts().to_string())
log("空值检查（Kaggle 拒绝含空值的文件）：", int(full.isna().sum().sum()))
log("queue_pred 取值集合：", sorted(full.loc[full.task == "queue", "queue_pred"].dropna().unique())[:6])
neg = full.loc[full.task == "odme", "path_flow"]
log("path_flow 最小值（应 >= 0）：", float(neg.min()) if len(neg) else "n/a")
log("完成。总耗时 %.1f 分钟" % ((time.time() - T0) / 60))


In [ ]:
# ============================================================
# 12. 线下自评：Task 1 在 train 上打真分
# ============================================================
# train 是唯一同时提供「掩码层」和「未掩码答案层」的 split，
# 所以 Task 1 是四个任务里唯一能在本地拿到真分数的。
# 这里用与官方 score_task1.py 相同的口径自评：
#   只评 is_score_eligible ∧ 被挖空 的格子；流量按【单车道】算 RMSE。
# 官方基线在 D12_I5_N 的 train 上 S_state = 0.6252，可用来对照。

EVAL_PANELS = None         # None = 全部走廊；想快就设 ["D12_I5_N"]
EVAL_SPLIT = "train"
EVAL_MAX_TRUTH_FILES = 0   # 0 = 全部；调小可加速


def _lanes_map(panel_dir):
    """station_id / link_id -> 车道数。流量按单车道计分，必须除它。"""
    m = {}
    for fn, key in (("links.csv", "link_id"),
                    ("fd_parameters.csv", "link_id"),
                    ("fd_parameters.csv", "station_id")):
        p = panel_dir / "network" / fn
        if not p.exists():
            continue
        d = pd.read_csv(p, dtype=str)
        if "lanes" not in d.columns or key not in d.columns:
            continue
        L = pd.to_numeric(d["lanes"], errors="coerce")
        ok = L.notna() & (L >= 1) & (L <= 16)      # 99 是区域连接线，绝不能当分母
        m.update(zip(d.loc[ok, key].astype(str), L[ok].astype(float)))
    return m


def eval_task1_panel(panel, split="train", max_truth_files=0):
    """在 train 上给一条走廊的 Task 1 打真分，并报告覆盖率。"""
    panel_dir = ROOT / "corridors" / panel
    net = network_params(panel_dir)
    mp = link_mileposts(panel_dir)
    prof_s, prof_f, prof_links = build_profiles(panel_dir, CFG)
    pred = predict_split(panel_dir, split, net, prof_s, prof_f, prof_links, mp, CFG)
    sub = apply_state_predictions(ROOT, panel, split, pred, CFG)
    if sub is None or not len(sub):
        log(f"  [跳过] {panel}: 没有生成 {split} 的预测")
        return None

    # 模板要求的行数 —— 衡量覆盖率。缺的行在官方计分里会按 0 进分母。
    tmpl_path = ROOT / "task1" / panel / split / "sample_submission_state.csv"
    n_required = len(pd.read_csv(tmpl_path, usecols=["panel"])) if tmpl_path.exists() else len(sub)

    sub["timestamp"] = pd.to_datetime(sub["timestamp"], utc=True)
    sub["station_id"] = sub.station_id.astype(str)
    sub["link_id"] = sub.link_id.astype(str)

    # 真值 = 未掩码观测层（只有 train 有）
    files = sorted((panel_dir / split / "mainline_states").glob("**/*.parquet"))
    if not files:
        log(f"  [跳过] {panel}: {split} 没有未掩码答案层，无法自评")
        return None
    if max_truth_files:
        files = files[:max_truth_files]
    parts = []
    for p in files:
        d = read_parquet_smart(p, ["timestamp", "station_id", "link_id",
                                   "speed_kmh", "flow_vph", "is_score_eligible"])
        if not len(d):
            continue
        d = as_link_str(as_ts(d))
        d = eligible_filter(d)
        if len(d):
            parts.append(d[["timestamp", "station_id", "link_id", "speed_kmh", "flow_vph"]])
    if not parts:
        return None
    truth = pd.concat(parts, ignore_index=True)
    truth["station_id"] = truth.station_id.astype(str)
    truth = truth.rename(columns={"speed_kmh": "true_speed", "flow_vph": "true_flow"})

    m = sub.merge(truth, on=["timestamp", "station_id", "link_id"], how="left")
    lanes = _lanes_map(panel_dir)
    ln = pd.to_numeric(m.station_id.map(lanes).fillna(m.link_id.map(lanes)),
                       errors="coerce").fillna(1.0).clip(lower=1.0).to_numpy()
    m["n_lanes"] = ln

    rows = []
    for reg in ("R1", "R2", "R3"):
        g = m[m.mask_regime == reg].dropna(subset=["true_speed", "true_flow"])
        if not len(g):
            rows.append(dict(panel=panel, regime=reg, n=0, rmse_speed=np.nan,
                             rmse_flow_lane=np.nan, S_speed=0.0, S_flow=0.0, S_state=0.0))
            continue
        ds = g["speed_kmh"].to_numpy(float) - g["true_speed"].to_numpy(float)
        dq = (g["flow_vph"].to_numpy(float) - g["true_flow"].to_numpy(float)) \
            / g["n_lanes"].to_numpy(float)
        rmse_s = float(np.sqrt(np.mean(ds * ds)))
        rmse_f = float(np.sqrt(np.mean(dq * dq)))
        s_sp = max(0.0, 1.0 - rmse_s / 25.0)
        s_fl = max(0.0, 1.0 - rmse_f / 600.0)
        rows.append(dict(panel=panel, regime=reg, n=len(g), rmse_speed=rmse_s,
                         rmse_flow_lane=rmse_f, S_speed=s_sp, S_flow=s_fl,
                         S_state=0.54 * s_sp + 0.46 * s_fl))
    det = pd.DataFrame(rows)
    return dict(panel=panel, coverage=len(sub) / max(n_required, 1),
                n_required=n_required, n_predicted=len(sub),
                n_matched=int(m["true_speed"].notna().sum()),
                S_state=float(det.S_state.mean()), detail=det)


eval_panels = EVAL_PANELS or panels
ev = []
for panel in eval_panels:
    log(f"[线下自评 Task1] {panel}")
    r = eval_task1_panel(panel, EVAL_SPLIT, EVAL_MAX_TRUTH_FILES)
    if r:
        ev.append(r)
        log(f"    S_state={r['S_state']:.4f}  覆盖率={r['coverage']*100:.2f}%  "
            f"({r['n_predicted']:,}/{r['n_required']:,})")

if ev:
    ev_df = pd.DataFrame([{k: v for k, v in r.items() if k != "detail"} for r in ev])
    print("\n=== Task 1 线下自评（train split）===")
    print(ev_df.to_string(index=False))
    print(f"\n平均 S_state = {ev_df.S_state.mean():.4f}")
    print(f"平均覆盖率   = {ev_df.coverage.mean() * 100:.2f}%")
    print("\n每个掩码档位的细节（第一条走廊）：")
    print(ev[0]["detail"].to_string(index=False))
    print("\n对照：官方历史均值基线在 D12_I5_N 的 train 上是 S_state = 0.6252")
else:
    print("没有可自评的走廊——请检查 train 的未掩码答案层是否存在。")


In [ ]:
# ============================================================
# 13. 诊断：Task 2 评估表为什么没出来
# ============================================================
# 只跑一条走廊、只读 60 天 train，三分钟出结果。
# 它会打印「窗口接受条件」的漏斗，一眼看出卡在哪一步。

DIAG_PANEL = "D12_I5_N"
DIAG_MAX_DAYS = 60

_pdg = ROOT / "corridors" / DIAG_PANEL
_netg = network_params(_pdg)
_smg = build_state_matrix(_pdg, "train", DIAG_MAX_DAYS)
if _smg is None:
    print(f"{DIAG_PANEL}: train 没有观测层")
else:
    SPg, OKg, FPg = _smg
    colsg = list(SPg.columns)
    # 和主流程保持一致：特征统计用全部 train（训练/推理同分布）
    ctg = queue_context(_pdg, _netg, colsg, SPg, OKg)
    Sg = SPg.to_numpy(float)
    Vg = OKg.to_numpy() & np.isfinite(Sg)
    Qg = (Sg <= ctg["vcut"][None, :]) & Vg
    ng = len(SPg.index)
    min_cov_g = effective_coverage(Vg, QUEUE_CFG)

    print("=" * 66)
    print(f"诊断走廊 {DIAG_PANEL}   train 文件上限 {DIAG_MAX_DAYS}")
    print(f"  时刻数 {ng:,}    路段数 {len(colsg)}")
    print(f"  v_cut 分位 [min,25,50,75,max] = "
          f"{np.percentile(ctg['vcut'], [0, 25, 50, 75, 100]).round(1)}")
    print(f"  有观测比例 V.mean() = {Vg.mean():.4f}"
          f"      ← 官方 min_hist_coverage = {QUEUE_CFG['min_hist_coverage']}")
    print(f"  实际采用的历史覆盖率下限 = {min_cov_g:.4f}")
    print(f"  排队格比例 Q.mean() = {Qg.mean():.5f}")
    print("=" * 66)

    _spacing = QUEUE_CFG["min_spacing_min"] // 5
    _lo = int(ng * (1.0 - QUEUE_CFG["holdout_frac"]))
    print(f"时间切分：前 {_lo:,} 步（约 {_lo/288:.0f} 天）只用于训练，"
          f"后 {ng-_lo:,} 步（约 {(ng-_lo)/288:.0f} 天）只用于评估")
    print("         —— 两段绝不重叠，否则模型分数会虚高（上一版就是这样）")
    n_cov = n_fut = n_both = 0
    _picked = {"queue_onset": [], "queue_ongoing": []}
    _last = {"queue_onset": -10 ** 9, "queue_ongoing": -10 ** 9}
    for _i0 in range(max(HIST_STEPS - 1, _lo), ng - FUT_STEPS):
        _vh = Vg[_i0 - HIST_STEPS + 1: _i0 + 1]
        _fu = Qg[_i0 + 1: _i0 + 1 + FUT_STEPS]
        _c = bool(_vh.size) and float(_vh.mean()) >= min_cov_g
        _f = bool(_fu.any())
        if _c:
            n_cov += 1
        if _f:
            n_fut += 1
        if _c and _f:
            n_both += 1
            _cond = "queue_ongoing" if Qg[_i0].any() else "queue_onset"
            if len(_picked[_cond]) < QUEUE_CFG["eval_per_cond"] and \
                    (_i0 - _last[_cond]) >= _spacing:
                _picked[_cond].append(_i0)
                _last[_cond] = _i0

    print(f"历史覆盖率达标 : {n_cov:,}")
    print(f"视野内有排队   : {n_fut:,}")
    print(f"两者同时满足   : {n_both:,}")
    print(f"按 360 分钟间隔最终选中 : onset {len(_picked['queue_onset'])} / "
          f"ongoing {len(_picked['queue_ongoing'])}")
    print("=" * 66)

    if any(len(v) for v in _picked.values()):
        def _diag_p(Xlink, meta):
            p = Xlink["queued_at_T"].to_numpy(float)[:, None]
            return np.repeat(p, FUT_STEPS, axis=1)

        def _diag_r(Xlink, meta):
            r = np.nan_to_num(Xlink["ratio_last"].to_numpy(float), nan=9.9)
            return np.repeat(np.where(r <= 1.15, 1.0, 0.0)[:, None], FUT_STEPS, axis=1)

        print("单走廊快速评估（自建窗口，已做时间切分，无泄漏）：")
        for _nm, _fn in (("persistence", _diag_p), ("rule", _diag_r)):
            _r = evaluate_windows(SPg, OKg, colsg, ctg, _picked, _fn, threshold=0.5, FP=FPg)
            if len(_r):
                _o = _r[_r.condition == "queue_onset"]["iou"].mean()
                _g = _r[_r.condition == "queue_ongoing"]["iou"].mean()
                print(f"  {_nm:12s} 等权平均 IoU = {_r['iou'].mean():.4f}"
                      f"    onset={_o:.3f}  ongoing={_g:.3f}")
        _qm = globals().get("q_model")
        _mp = globals().get("_model_prob")
        if _qm is not None and _mp is not None:
            _thr = globals().get("q_thr") or {"queue_onset": 0.5, "queue_ongoing": 0.5}
            _rows = []
            for _c, _t in _thr.items():
                if _picked.get(_c):
                    _r1 = evaluate_windows(SPg, OKg, colsg, ctg, {_c: _picked[_c]},
                                           _mp, threshold=_t, FP=FPg)
                    if len(_r1):
                        _rows.append(_r1)
            if _rows:
                _rm = pd.concat(_rows, ignore_index=True)
                _mo = _rm[_rm.condition == "queue_onset"]["iou"].mean()
                _mg = _rm[_rm.condition == "queue_ongoing"]["iou"].mean()
                print(f"  {'model':12s} 等权平均 IoU = {_rm['iou'].mean():.4f}"
                      f"    onset={_mo:.3f}  ongoing={_mg:.3f}"
                      f"    (阈值 onset={_thr['queue_onset']:.2f} / "
                      f"ongoing={_thr['queue_ongoing']:.2f})")
        else:
            print("  model        未通过闸门 / 未训练，故未纳入")
    else:
        print("『最终选中』是 0 —— 没有窗口可评，所以评估表不会打印。")
        print("请把上面几行数字发我，我据此调整窗口接受条件。")
